# FAQ Chatbot - Retrieval Analysis

An end-to-end look at the FAQ retrieval pipeline: preprocessing, TF-IDF
vectorization, cosine similarity and threshold-based fallback behaviour.

## 1. Imports and configuration

In [ ]:
from src.config import settings
from src.data_loading import load_faq_dataset, load_test_questions
from src.preprocessing import TextPreprocessor
from src.vectorizer import TfidfTextVectorizer
from src.retrieval import FAQRetriever
from src.chatbot import FAQChatbot
from src.evaluation import evaluate, threshold_sweep, plot_threshold_analysis

import pandas as pd
pd.set_option('display.max_colwidth', 70)

## 2. The FAQ knowledge base

In [ ]:
faqs = load_faq_dataset(settings.resolve_faq_dataset())
print(f'{len(faqs)} FAQ entries across {faqs['category'].nunique()} categories')
faqs.groupby('category').size()

## 3. Text preprocessing in action

In [ ]:
pp = TextPreprocessor(normalization=settings.normalization)
for q in ["I can't log into my account!", "How do I get my money back?"]:
    print(f'{q!r:45} -> {pp.transform(q)}')

## 4. Build the retriever and try a few queries

In [ ]:
chatbot = FAQChatbot.from_settings(settings)
retriever = chatbot.retriever
print(f'vocabulary size: {retriever.vectorizer.vocabulary_size}')

queries = ["I forgot my password", "How do I cancel my order?", "Tell me a joke"]
for q in queries:
    r = chatbot.respond(q)
    print(f'Q: {q!r}')
    print(f'   fallback={r.is_fallback}  faq={r.faq_id}  confidence={r.confidence:.3f}')
    print(f'   answer: {r.answer[:80]}...')

## 5. Offline evaluation on the held-out test questions

In [ ]:
test_df = load_test_questions(settings.resolve_test_questions())
result = evaluate(retriever, test_df, settings.confidence_threshold, faqs=faqs)
m = result['metrics']
print(f"Top-1 Accuracy : {m['top1_accuracy']:.1%}")
print(f"Top-3 Accuracy : {m['top3_accuracy']:.1%}")
print(f"MRR            : {m['mrr']:.1%}")
print(f"Fallback rate  : {m['fallback_rate']:.1%}")
print(f"OOS catch rate : {m['out_of_scope_catch_rate']:.1%}")

## 6. Choosing the confidence threshold

In [ ]:
sweep = threshold_sweep(retriever, test_df, faqs=faqs)
plot_threshold_analysis(sweep, settings.resolve_reports_dir() / 'threshold_analysis.png')
sweep[['threshold', 'end_to_end_accuracy', 'fallback_rate', 'out_of_scope_catch_rate']].head(12)

Higher thresholds reject more (fallback rate rises) and catch more
out-of-scope questions, but also refuse valid in-scope questions. The
default operating point balances coverage against wrong-answer risk.

## 7. Where the lexical baseline fails

In [ ]:
preds = result['predictions']
def in_expanded(best_id, expected):
    return bool(best_id) and best_id in (expected or ())
mis = preds[(preds['out_of_scope'] == False) & (preds['answered'] == True)]
mis = mis[~mis.apply(lambda r: in_expanded(r['best_id'], r['expected_expanded']), axis=1)]
mis[['question', 'best_id', 'best_score']].head(10)

These are typically *synonym* mismatches ("card rejected" vs "card
declined") or short queries whose only overlapping token is generic
("account", "order"). Sentence embeddings solve most of them; see
`docs/error_analysis.md` for a detailed breakdown.